# Assignment 02 — Named Entity Recognition (NER) from a News Article

**Course:** AIAC 536 — Natural Language Processing
**Programme:** MTech in Artificial Intelligence, Kathmandu University
**Student:** Prasanna Koirala

---

## What this assignment asks for

1. Take an English news article from a reputable source, saved as `.txt`
2. Load it into Python
3. Perform **Named Entity Recognition (NER)**
4. Extract all named entities **and their types** (PERSON, ORG, GPE, DATE, MONEY, …)
5. Save them to a CSV with columns `Entity`, `Entity_Type`
6. **Provide a summary of the different entity types found**

Plus two optional extras, which we do as well:

- Visualise the entities using spaCy's `displacy`
- **Compare two different NER models** and discuss the results

## What is Named Entity Recognition?

In Assignment 01 we asked of every word: **"what grammatical job does this do?"**
That was POS tagging.

NER asks a completely different question: **"what real-world thing is this?"**

Take a sentence:

> NASA launched a rocket from Kennedy Space Center in Florida on Sunday.

NER ignores most of it and picks out only the **things that have names**:

| Found | Type | Meaning |
|---|---|---|
| NASA | `ORG` | an organisation |
| Kennedy Space Center | `FAC` | a building or facility |
| Florida | `GPE` | a place (country/city/state) |
| Sunday | `DATE` | a date |

It does not care about "launched", "rocket", or "the". Those are not *named things*.

### Why this matters

If you want to answer questions like *who was involved, where did it happen, when, and
how much did it cost* — NER is the tool that pulls those out of plain text. It powers
search, news aggregation, CV screening, and document processing.

## Step 0 — Setup

In [1]:
# --- Python's own built-in tools ---
from collections import Counter   # counts how many times each item appears
from pathlib import Path          # safe, readable file paths

# --- Third-party libraries ---
import pandas as pd               # builds the results table and writes the CSV
import spacy                      # the NLP library that performs NER
from spacy import displacy        # draws the coloured entity visualisation

# spacy.explain() warns on a few tags with no glossary entry. Harmless - hide it.
import warnings
warnings.filterwarnings("ignore", message=r".*W118.*")

# We load TWO models this time, because one of the optional tasks is to
# compare two different NER models.
#
#   en_core_web_sm  - "small"  (~12 MB) - the one we used in Assignment 01
#   en_core_web_md  - "medium" (~40 MB) - same design, more training data,
#                     and it includes real word vectors
nlp_sm = spacy.load("en_core_web_sm")
nlp_md = spacy.load("en_core_web_md")

print("spaCy version :", spacy.__version__)
print("Model 1       :", nlp_sm.meta["name"], "v" + nlp_sm.meta["version"])
print("Model 2       :", nlp_md.meta["name"], "v" + nlp_md.meta["version"])
print()
print("Pipeline steps:", nlp_sm.pipe_names)

spaCy version : 3.8.16
Model 1       : core_web_sm v3.8.0
Model 2       : core_web_md v3.8.0

Pipeline steps: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


Notice the pipeline includes a component called **`ner`**. That is the piece doing the
work in this assignment — the same pipeline that gave us `tagger` for Assignment 01.

## Step 1 — See NER work on a single sentence

As before, start small enough to check every result by eye.

In [2]:
demo = nlp_sm("NASA launched a rocket from Kennedy Space Center in Florida on Sunday. "
              "The $500 million mission was led by Nicky Fox.")

# doc.ents holds the named entities spaCy found.
# Each one has:
#   ent.text   -> the words that make it up
#   ent.label_ -> the entity type (ORG, GPE, DATE, ...)
print(f"{'ENTITY':<24} {'TYPE':<10} MEANING")
print("-" * 66)
for ent in demo.ents:
    print(f"{ent.text:<24} {ent.label_:<10} {spacy.explain(ent.label_)}")

print()
print(f"Words in the sentence : {len([t for t in demo if not t.is_punct])}")
print(f"Entities found        : {len(demo.ents)}")

ENTITY                   TYPE       MEANING
------------------------------------------------------------------
NASA                     ORG        Companies, agencies, institutions, etc.
Kennedy Space Center     FAC        Buildings, airports, highways, bridges, etc.
Florida                  GPE        Countries, cities, states
Sunday                   DATE       Absolute or relative dates or periods
$500 million             MONEY      Monetary values, including unit
Nicky Fox                PERSON     People, including fictional

Words in the sentence : 22
Entities found        : 6


**Only 6 things found out of 22 words.** That is the essential difference from
Assignment 01, where every single word received a label.

## Step 2 — An entity can be more than one word

This is a structural difference from POS tagging, and it matters for the code.

POS tagging labels **one word at a time**. NER can group **several words into a single
thing**.

In [3]:
# len(ent) tells us how many tokens make up this one entity.
print(f"{'ENTITY':<24} {'TOKENS':<8} MADE UP OF")
print("-" * 62)
for ent in demo.ents:
    pieces = " + ".join(token.text for token in ent)
    print(f"{ent.text:<24} {len(ent):<8} {pieces}")

ENTITY                   TOKENS   MADE UP OF
--------------------------------------------------------------
NASA                     1        NASA
Kennedy Space Center     3        Kennedy + Space + Center
Florida                  1        Florida
Sunday                   1        Sunday
$500 million             3        $ + 500 + million
Nicky Fox                2        Nicky + Fox


`Kennedy Space Center` is **three words but one entity**. So is `$500 million`. Splitting
them apart would destroy the meaning — "Kennedy" alone is a person, not a facility.

This is why the CSV for this assignment has one row per **entity**, not per word.

## Step 3 — Load the article

We deliberately reuse **the same NASA article as Assignment 01**. That lets us compare
the two techniques directly on identical text, which is far more informative than using
two unrelated articles.

In [4]:
ARTICLE_PATH = Path("data/article.txt")
article_text = ARTICLE_PATH.read_text(encoding="utf-8")

print("File loaded :", ARTICLE_PATH)
print("Characters  :", len(article_text))
print("Words (rough):", len(article_text.split()))
print()
print(article_text[:300], "...")

File loaded : data/article.txt
Characters  : 5559
Words (rough): 843

NASA's Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches

Now on a three-month, million-mile journey to its final orbit, NASA's Nancy Grace Roman Space Telescope will soon reveal the universe's darkest secrets. The mission launched at 7:26 a.m. EDT Sunday aboard a SpaceX Falcon Heavy  ...


## Step 4 — POS vs NER on the same words

This is the clearest way to see what NER adds. We take one sentence and show what each
technique says about every word.

In [5]:
sentence = nlp_sm("NASA launched a rocket from Kennedy Space Center in Florida on Sunday.")

# Build a lookup: which entity type does each token belong to (if any)?
# ent.start .. ent.end are token positions, so we can walk them.
token_entity = {}
for ent in sentence.ents:
    for token in ent:
        token_entity[token.i] = ent.label_

print(f"{'WORD':<16} {'POS says (grammar)':<22} {'NER says (real world)'}")
print("-" * 62)
for token in sentence:
    if token.is_punct:
        continue
    # "-" means this word is not part of any named entity
    entity_type = token_entity.get(token.i, "-")
    print(f"{token.text:<16} {token.pos_:<22} {entity_type}")

WORD             POS says (grammar)     NER says (real world)
--------------------------------------------------------------
NASA             PROPN                  ORG
launched         VERB                   -
a                DET                    -
rocket           NOUN                   -
from             ADP                    -
Kennedy          PROPN                  FAC
Space            PROPN                  FAC
Center           PROPN                  FAC
in               ADP                    -
Florida          PROPN                  GPE
on               ADP                    -
Sunday           PROPN                  DATE


### The key insight

Look carefully at the rows for **NASA, Kennedy, Space, Center, Florida** and **Sunday**.

**POS gave every one of them the identical label: `PROPN`** (proper noun). POS can tell
you *"this is a name"* — but nothing more. It cannot distinguish an organisation from a
country from a date.

**NER tells you which kind of name it is:** `ORG`, `FAC`, `GPE`, `DATE`.

So NER is essentially a **refinement of `PROPN`**. It takes the words POS lumped together
as "names" and identifies what each one actually is.

This connects the two assignments directly. In Assignment 01 we found that **`PROPN` made
up 43% of all nouns** in this article — a large pile of names that POS could not tell
apart. NER is the tool that sorts them out.

## Step 5 — Run NER on the whole article

In [6]:
# One call does everything: tokenise, tag, and find entities.
doc = nlp_sm(article_text)

print("Entities found:", len(doc.ents))
print()
print("First 20:")
print(f"  {'ENTITY':<44} {'TYPE'}")
print("  " + "-" * 56)
for ent in doc.ents[:20]:
    print(f"  {ent.text:<44} {ent.label_}")

Entities found: 116

First 20:
  ENTITY                                       TYPE
  --------------------------------------------------------
  NASA                                         ORG
  Dark Universe-Seeking                        PERSON
  Nancy Grace Roman Space                      PERSON
  three-month                                  DATE
  million-mile                                 QUANTITY
  NASA                                         ORG
  Nancy Grace Roman Space Telescope            PERSON
  7:26 a.m. EDT                                TIME
  Sunday                                       DATE
  Falcon Heavy                                 ORG
  Launch                                       NORP
  Kennedy Space Center                         FAC
  Florida                                      GPE
  Roman                                        NORP
  Roman                                        PERSON
  NASA                                         ORG
  NASA              

## Step 6 — What the entity types mean

spaCy's English models recognise 18 entity types. Here are the ones that appear in this
article, with plain-English meanings.

In [7]:
# Which types actually occur here, and how often?
type_counts = Counter(ent.label_ for ent in doc.ents)

print(f"{'TYPE':<12} {'COUNT':<7} MEANING")
print("-" * 70)
for label, count in type_counts.most_common():
    print(f"{label:<12} {count:<7} {spacy.explain(label)}")

TYPE         COUNT   MEANING
----------------------------------------------------------------------
ORG          45      Companies, agencies, institutions, etc.
GPE          13      Countries, cities, states
PERSON       12      People, including fictional
DATE         10      Absolute or relative dates or periods
NORP         10      Nationalities or religious or political groups
TIME         6       Times smaller than a day
CARDINAL     6       Numerals that do not fall under another type
ORDINAL      5       "first", "second", etc.
LOC          4       Non-GPE locations, mountain ranges, bodies of water
FAC          3       Buildings, airports, highways, bridges, etc.
QUANTITY     2       Measurements, as of weight or distance


In [8]:
# Show a real example of each type from the article, so the labels are concrete.
print("ONE REAL EXAMPLE OF EACH TYPE")
print("=" * 62)

seen = set()
for ent in doc.ents:
    if ent.label_ not in seen:
        seen.add(ent.label_)
        print(f"  {ent.label_:<12} {ent.text}")

ONE REAL EXAMPLE OF EACH TYPE
  ORG          NASA
  PERSON       Dark Universe-Seeking
  DATE         three-month
  QUANTITY     million-mile
  TIME         7:26 a.m. EDT
  NORP         Launch
  FAC          Kennedy Space Center
  GPE          Florida
  ORDINAL      second
  CARDINAL     L2
  LOC          Earth


## Step 7 — Building the results table

One row per entity found. The brief requires `Entity` and `Entity_Type`; the extra
columns cost nothing and support the analysis.

In [9]:
# Work out which sentence each entity belongs to, so we can trace it back.
sentences = list(doc.sents)

rows = []
for ent in doc.ents:
    # Find the sentence number this entity starts in.
    sentence_number = None
    for i, sent in enumerate(sentences, start=1):
        if sent.start_char <= ent.start_char < sent.end_char:
            sentence_number = i
            break

    rows.append({
        "Entity":       ent.text,                      # the words themselves
        "Entity_Type":  ent.label_,                    # ORG, GPE, DATE, ...
        "Description":  spacy.explain(ent.label_),     # plain-English meaning
        "Token_Count":  len(ent),                      # how many words it spans
        "Sentence_No":  sentence_number,               # where it came from
    })

ner_df = pd.DataFrame(rows)

print("Rows:", len(ner_df))
ner_df.head(15)

Rows: 116


,Entity,Entity_Type,Description,Token_Count,Sentence_No
0,NASA,ORG,"Companies, agencies, institutions, etc.",1,1
1,Dark Universe-Seeking,PERSON,"People, including fictional",4,1
2,Nancy Grace Roman Space,PERSON,"People, including fictional",4,1
3,three-month,DATE,Absolute or relative dates or periods,3,1
4,million-mile,QUANTITY,"Measurements, as of weight or distance",3,1
5,NASA,ORG,"Companies, agencies, institutions, etc.",1,1
6,Nancy Grace Roman Space Telescope,PERSON,"People, including fictional",5,1
7,7:26 a.m. EDT,TIME,Times smaller than a day,3,2
8,Sunday,DATE,Absolute or relative dates or periods,1,2
9,Falcon Heavy,ORG,"Companies, agencies, institutions, etc.",2,2


## Step 8 — Summary of entity types

**This is a required deliverable**, not an extra. The brief asks for "a summary of the
different entity types found".

In [10]:
# Count entities by type, with percentages.
summary = (
    ner_df.groupby("Entity_Type")
          .agg(Count=("Entity", "size"),
               Unique=("Entity", "nunique"))       # how many DISTINCT entities
          .reset_index()
          .sort_values("Count", ascending=False)
)
summary["Percent"] = (100 * summary["Count"] / len(ner_df)).round(1)
summary["Meaning"] = summary["Entity_Type"].apply(spacy.explain)

print(f"ENTITY TYPE SUMMARY  -  {len(ner_df)} entities, "
      f"{ner_df['Entity'].nunique()} unique, {len(summary)} types")
summary

ENTITY TYPE SUMMARY  -  116 entities, 79 unique, 11 types


,Entity_Type,Count,Unique,Percent,Meaning
7,ORG,45,25,38.8,"Companies, agencies, institutions, etc."
3,GPE,13,12,11.2,"Countries, cities, states"
8,PERSON,12,9,10.3,"People, including fictional"
1,DATE,10,9,8.6,Absolute or relative dates or periods
5,NORP,10,3,8.6,Nationalities or religious or political groups
0,CARDINAL,6,6,5.2,Numerals that do not fall under another type
10,TIME,6,6,5.2,Times smaller than a day
6,ORDINAL,5,3,4.3,"""first"", ""second"", etc."
4,LOC,4,3,3.4,"Non-GPE locations, mountain ranges, bodies of ..."
2,FAC,3,3,2.6,"Buildings, airports, highways, bridges, etc."


In [11]:
# A simple text bar chart, which reads more easily than a table of numbers.
print("ENTITY TYPES BY FREQUENCY")
print("=" * 56)
for _, row in summary.iterrows():
    bar = "#" * row["Count"]
    print(f"  {row['Entity_Type']:<11} {row['Count']:>3}  {bar}")

ENTITY TYPES BY FREQUENCY
  ORG          45  #############################################
  GPE          13  #############
  PERSON       12  ############
  DATE         10  ##########
  NORP         10  ##########
  CARDINAL      6  ######
  TIME          6  ######
  ORDINAL       5  #####
  LOC           4  ####
  FAC           3  ###
  QUANTITY      2  ##


In [12]:
# The most frequently mentioned entities overall - what the article is about.
print("MOST MENTIONED ENTITIES")
print("=" * 48)
for entity, count in Counter(ner_df["Entity"]).most_common(12):
    entity_type = ner_df[ner_df["Entity"] == entity]["Entity_Type"].iloc[0]
    print(f"  {entity:<28} {entity_type:<10} x{count}")

MOST MENTIONED ENTITIES
  Roman                        NORP       x20
  NASA                         ORG        x11
  Falcon Heavy                 ORG        x3
  first                        ORDINAL    x3
  three-month                  DATE       x2
  Earth                        LOC        x2
  California                   GPE        x2
  NASA Goddard                 ORG        x2
  Dark Universe-Seeking        PERSON     x1
  Nancy Grace Roman Space      PERSON     x1
  million-mile                 QUANTITY   x1
  Nancy Grace Roman Space Telescope PERSON     x1


### Reading the summary

Numbers alone are not a summary — here is what they actually say about this article.

**`ORG` dominates at 38.8%** — 45 of 116 entities. That is what a mission announcement
looks like: NASA, SpaceX, ESA, JAXA, CNES, BAE Systems, L3Harris, Caltech. Space projects
are built by consortia, and the text is thick with institutional names.

**`GPE` and `LOC` together account for 14.6%**, because the article physically tours the
world — Florida, Maryland, Washington, Australia, Spain, California, Germany. That is the
Deep Space Network handing off between ground stations, and the entity types trace the
route.

**`PERSON` is only 10.3%, and just 9 unique people.** For a news article that is low.
This is a story about *hardware and institutions*, not personalities — the three people
quoted are there to speak on behalf of organisations.

**`NORP` is suspicious.** 10 mentions but only **3 unique** values. A nationality tag
appearing repeatedly in an article with no nationalities discussed is a red flag, and
Step 11 identifies the culprit: the word "Roman".

**The `Unique` column is doing real work here.** `ORG` has 45 mentions but 25 unique
values, so organisations are genuinely varied. `NORP` has 10 mentions from 3 values — the
same handful repeated. Raw counts alone would hide that difference entirely.

## Step 9 — Export to CSV

In [13]:
OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)
CSV_PATH = OUTPUT_DIR / "PrasannaKoirala_NER_01.csv"

ner_df.to_csv(CSV_PATH, index=False, encoding="utf-8")

print("Saved:", CSV_PATH)
print("Rows :", len(ner_df))
print("Size :", CSV_PATH.stat().st_size, "bytes")

Saved: outputs/PrasannaKoirala_NER_01.csv
Rows : 116
Size : 7195 bytes


In [14]:
# Read it back off disk and verify. Never trust an unverified write.
check = pd.read_csv(CSV_PATH)

print("Rows read back :", len(check))
print("Columns        :", list(check.columns))
print("Empty cells    :", check.isna().sum().sum())
print()
check.head()

Rows read back : 116
Columns        : ['Entity', 'Entity_Type', 'Description', 'Token_Count', 'Sentence_No']
Empty cells    : 0



,Entity,Entity_Type,Description,Token_Count,Sentence_No
0,NASA,ORG,"Companies, agencies, institutions, etc.",1,1
1,Dark Universe-Seeking,PERSON,"People, including fictional",4,1
2,Nancy Grace Roman Space,PERSON,"People, including fictional",4,1
3,three-month,DATE,Absolute or relative dates or periods,3,1
4,million-mile,QUANTITY,"Measurements, as of weight or distance",3,1


In [15]:
# Also save the type summary, since the brief asks for a summary as a deliverable.
SUMMARY_PATH = OUTPUT_DIR / "PrasannaKoirala_NER_01_summary.csv"
summary.to_csv(SUMMARY_PATH, index=False, encoding="utf-8")
print("Saved:", SUMMARY_PATH)
summary[["Entity_Type", "Count", "Unique", "Percent"]]

Saved: outputs/PrasannaKoirala_NER_01_summary.csv


,Entity_Type,Count,Unique,Percent
7,ORG,45,25,38.8
3,GPE,13,12,11.2
8,PERSON,12,9,10.3
1,DATE,10,9,8.6
5,NORP,10,3,8.6
0,CARDINAL,6,6,5.2
10,TIME,6,6,5.2
6,ORDINAL,5,3,4.3
4,LOC,4,3,3.4
2,FAC,3,3,2.6


## Step 10 — Bonus: visualising the entities

`displacy` is spaCy's built-in visualiser. It renders the text with each entity
highlighted and colour-coded by type, which makes errors far easier to spot than reading
a table.

We render the first few sentences only, to keep the notebook a sensible size.

In [16]:
# Take just the first 3 sentences for the visualisation.
first_sentences = list(doc.sents)[:3]

# Build a small Doc containing only that span of text.
start = first_sentences[0].start_char
end = first_sentences[-1].end_char
snippet = nlp_sm(article_text[start:end])

# style="ent" draws entities. jupyter=True renders it inline in the notebook.
displacy.render(snippet, style="ent", jupyter=True)

Reading the colours is the quickest way to sanity-check NER output. Two things are worth
noticing in the rendering above, and we investigate them properly in the next step.

## Step 11 — Bonus: comparing two NER models

The brief suggests comparing two different NER models. We compare spaCy's **small** and
**medium** English models.

Why these two? They share the same architecture and the same 18 entity labels, so the
comparison is genuinely like-for-like. The only meaningful difference is that `md` is
trained with more data and carries real word vectors. Any disagreement is therefore
attributable to model capacity, not to differing label schemes.

In [17]:
# Run the medium model over the same article.
doc_md = nlp_md(article_text)

print(f"{'':<26} {'small':>8} {'medium':>8}")
print("-" * 46)
print(f"{'Entities found':<26} {len(doc.ents):>8} {len(doc_md.ents):>8}")
print(f"{'Unique entity texts':<26} "
      f"{len({e.text for e in doc.ents}):>8} {len({e.text for e in doc_md.ents}):>8}")
print(f"{'Entity types used':<26} "
      f"{len({e.label_ for e in doc.ents}):>8} {len({e.label_ for e in doc_md.ents}):>8}")

                              small   medium
----------------------------------------------
Entities found                  116      116
Unique entity texts              79       79
Entity types used                11       12


In [18]:
# Compare the type breakdown side by side.
counts_sm = Counter(e.label_ for e in doc.ents)
counts_md = Counter(e.label_ for e in doc_md.ents)

comparison = pd.DataFrame([
    {"Entity_Type": label,
     "Small": counts_sm.get(label, 0),
     "Medium": counts_md.get(label, 0),
     "Difference": counts_md.get(label, 0) - counts_sm.get(label, 0)}
    for label in sorted(set(counts_sm) | set(counts_md))
]).sort_values("Small", ascending=False)

comparison

,Entity_Type,Small,Medium,Difference
7,ORG,45,50,5
3,GPE,13,14,1
8,PERSON,12,8,-4
1,DATE,10,9,-1
5,NORP,10,8,-2
0,CARDINAL,6,5,-1
10,TIME,6,6,0
6,ORDINAL,5,5,0
4,LOC,4,4,0
2,FAC,3,3,0


The **totals are nearly identical**, which is a trap. Equal counts do not mean equal
answers — the two models can find the same *number* of entities while disagreeing about
*which* ones. So we compare them properly, by position in the text.

In [19]:
# Each entity occupies a character range in the article. Two entities are "the same
# entity" only if they cover the same characters. Comparing by position like this is
# reliable; comparing by list index is not.
spans_sm = {(e.start_char, e.end_char): e.label_ for e in doc.ents}
spans_md = {(e.start_char, e.end_char): e.label_ for e in doc_md.ents}

both      = spans_sm.keys() & spans_md.keys()
only_sm   = spans_sm.keys() - spans_md.keys()
only_md   = spans_md.keys() - spans_sm.keys()
same_label = [k for k in both if spans_sm[k] == spans_md[k]]
diff_label = [k for k in both if spans_sm[k] != spans_md[k]]

print("COMPARING BY POSITION IN THE TEXT")
print("=" * 52)
print(f"  Found by both, same label      : {len(same_label)}")
print(f"  Found by both, DIFFERENT label : {len(diff_label)}")
print(f"  Found only by small            : {len(only_sm)}")
print(f"  Found only by medium           : {len(only_md)}")
print("-" * 52)
print(f"  Full agreement                 : "
      f"{len(same_label) / len(spans_sm | spans_md):.1%}")

COMPARING BY POSITION IN THE TEXT
  Found by both, same label      : 90
  Found by both, DIFFERENT label : 10
  Found only by small            : 16
  Found only by medium           : 16
----------------------------------------------------
  Full agreement                 : 68.2%


In [20]:
# The interesting cases: both models found the same words, but disagreed on the type.
disagreements = pd.DataFrame([
    {"Entity": article_text[start:end],
     "Small_Says": spans_sm[(start, end)],
     "Medium_Says": spans_md[(start, end)]}
    for start, end in diff_label
])

print("SAME WORDS, DIFFERENT TYPE")
disagreements

SAME WORDS, DIFFERENT TYPE


,Entity,Small_Says,Medium_Says
0,Roman,PERSON,NORP
1,Caltech,GPE,ORG
2,Roman,NORP,ORG
3,Roman,PERSON,ORG
4,Goddard Space Flight Center,ORG,FAC
5,Roman,NORP,ORG
6,the Canberra Deep Space Communication Complex,FAC,ORG
7,Roman,NORP,ORG
8,Roman,PERSON,NORP
9,18 4K,CARDINAL,QUANTITY


### What the comparison shows

The two models agree on the overall shape of the article but disagree in detail, and the
disagreements are not evenly spread — they concentrate on **one particular word**.

Look at how often **"Roman"** appears in the disagreement table.

That word is genuinely hard, and for an interesting reason. The telescope is named after
**Nancy Grace Roman**, an astronomer. So "Roman" is simultaneously:

- a **person's surname** (`PERSON`)
- the **name of a mission/instrument** (`ORG`)
- and — unhelpfully — an ordinary English word meaning *from Rome* (`NORP`, which covers
  nationalities)

Neither model is reliably right, and worse, **each model tags "Roman" differently in
different sentences of the same article.** A model can be internally inconsistent.

This is a much more useful finding than "the bigger model is better". The honest
conclusion is that **model size did not resolve the genuinely ambiguous case** — it is
ambiguous in the text itself, not merely under-modelled.

In [21]:
# Track every occurrence of "Roman" and see what each model called it.
print("EVERY 'Roman' IN THE ARTICLE")
print("=" * 58)
print(f"  {'#':<4} {'small':<10} {'medium':<10}")
print("  " + "-" * 30)

roman_sm = [e for e in doc.ents if e.text == "Roman"]
roman_md = {(e.start_char, e.end_char): e.label_ for e in doc_md.ents}

for i, ent in enumerate(roman_sm, start=1):
    md_label = roman_md.get((ent.start_char, ent.end_char), "not found")
    flag = "" if md_label == ent.label_ else "   <-- differ"
    print(f"  {i:<4} {ent.label_:<10} {md_label:<10}{flag}")

print()
print(f"  Labels the SMALL model gave to 'Roman' : "
      f"{sorted({e.label_ for e in roman_sm})}")

EVERY 'Roman' IN THE ARTICLE
  #    small      medium    
  ------------------------------
  1    NORP       NORP      
  2    PERSON     NORP         <-- differ
  3    ORG        ORG       
  4    NORP       ORG          <-- differ
  5    NORP       ORG          <-- differ
  6    ORG        ORG       
  7    NORP       NORP      
  8    NORP       NORP      
  9    NORP       NORP      
  10   ORG        ORG       
  11   ORG        not found    <-- differ
  12   ORG        ORG       
  13   NORP       NORP      
  14   ORG        ORG       
  15   ORG        ORG       
  16   NORP       ORG          <-- differ
  17   PERSON     PERSON    
  18   ORG        ORG       
  19   PERSON     NORP         <-- differ
  20   PERSON     ORG          <-- differ

  Labels the SMALL model gave to 'Roman' : ['NORP', 'ORG', 'PERSON']


## Step 12 — Error analysis, part 1: the headline

Reading the `displacy` output above, two entities near the very start look wrong. They
are worth chasing down, because the cause is a general problem, not a one-off.

In [22]:
# Run NER on the headline BY ITSELF and inspect what comes back.
headline = article_text.splitlines()[0]

print("HEADLINE:")
print(f"  {headline}")
print()
print("What the model finds in it:")
for ent in nlp_sm(headline).ents:
    print(f"  {ent.text:<34} {ent.label_}")

HEADLINE:
  NASA's Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches

What the model finds in it:
  NASA                               ORG
  Dark Universe-Seeking              PERSON
  Nancy Grace Roman Space            PERSON


Three entities, and **two of them are wrong**:

| Found | Labelled | Verdict |
|---|---|---|
| NASA | `ORG` | correct |
| Dark Universe-Seeking | `PERSON` | **wrong** — it is a descriptive phrase, not a person |
| Nancy Grace Roman Space | `PERSON` | **wrong span** — it truncates "…Space Telescope" |

"Dark Universe-Seeking" is not a name of anything. The model invented an entity.

### Why headlines break NER

Headlines are not sentences. This one has:

- **Title Case On Almost Every Word** — and capitalisation is one of the strongest signals
  a model uses to spot names. A headline makes ordinary words look like names.
- **No main verb** — "Launches" is doing the work of a verb but reads as a plural noun.
- **Compression** — "Dark Universe-Seeking" is a stacked modifier, a construction that
  barely occurs in ordinary prose.

NER models are trained overwhelmingly on **running prose**. Headlines are a different
genre, and the model is effectively out of its training distribution.

In [23]:
# Test the theory: write the same facts as an ordinary sentence and re-run.
rewritten = ("NASA launched the Nancy Grace Roman Space Telescope, "
             "which seeks to study the dark universe.")

print("Same facts, ordinary sentence:")
print(f"  {rewritten}")
print()
print("What the model finds now:")
for ent in nlp_sm(rewritten).ents:
    print(f"  {ent.text:<34} {ent.label_}")

Same facts, ordinary sentence:
  NASA launched the Nancy Grace Roman Space Telescope, which seeks to study the dark universe.

What the model finds now:


  NASA                               ORG


### The result is not what you would expect

The hallucinated "Dark Universe-Seeking" **is gone**, which supports the theory.

But the model now finds only `NASA` — it **misses the telescope entirely**. So rewriting
did not fix the problem; it swapped one error for the opposite one:

| | Headline | Ordinary sentence |
|---|---|---|
| Failure mode | **over-detects** — invents entities | **under-detects** — misses a real one |

Neither version is correct. This is worth stating plainly rather than claiming the
sentence version "fixed" it.

In [24]:
# A second, separate problem: the headline has no full stop.
# See what spaCy therefore treats as "sentence 1".
sentence_1 = list(doc.sents)[0]

print("Headline ends with:", repr(headline[-12:]))
print("  -> no full stop, so the sentence splitter does not break here.")
print()
print("What spaCy calls SENTENCE 1:")
print(f"  {sentence_1.text[:200]!r} ...")
print()
print("Entities the model assigns to that merged 'sentence':")
for ent in doc.ents:
    if ent.start_char < sentence_1.end_char:
        print(f"  {ent.text:<36} {ent.label_}")

Headline ends with: 'ope Launches'
  -> no full stop, so the sentence splitter does not break here.

What spaCy calls SENTENCE 1:
  "NASA's Dark Universe-Seeking Nancy Grace Roman Space Telescope Launches\n\nNow on a three-month, million-mile journey to its final orbit, NASA's Nancy Grace Roman Space Telescope will soon reveal the un" ...

Entities the model assigns to that merged 'sentence':
  NASA                                 ORG
  Dark Universe-Seeking                PERSON
  Nancy Grace Roman Space              PERSON
  three-month                          DATE
  million-mile                         QUANTITY
  NASA                                 ORG
  Nancy Grace Roman Space Telescope    PERSON


### A second, independent bug: sentence segmentation

The headline ends in "Launches" with **no full stop**. Sentence splitters break on
punctuation, so spaCy never breaks there — it glues the headline onto the first paragraph
and calls the whole thing one sentence.

Consequences:

1. Our `Sentence_No` column is skewed — sentence 1 is really *headline + paragraph 1*.
2. The model reads headline text and body text as one continuous unit, which is exactly
   the genre-mixing that produced the bad tags in the first place.

**The fix would be a preprocessing step**: separate the headline from the body before
tagging, and treat them differently. We have deliberately not done that here — the
assignment says to load and process the article as saved, and silently editing the source
text would misrepresent what the model actually does on real input. Documenting the
limitation is the more honest choice.

### The deepest problem: the label set has no right answer

Even in clean body prose, the model tags **"Nancy Grace Roman Space Telescope"** as
`PERSON`.

Before calling that a mistake, consider: the telescope **is named after a person** —
Nancy Grace Roman, NASA's first Chief of Astronomy. So `PERSON` is not absurd.

But what *should* it be? Run through spaCy's 18 types:

- `PERSON` — it is named after one, but it is a machine
- `ORG` — it is not an organisation
- `PRODUCT` — closest, but built once for science, not a product
- `FAC` — facilities are buildings; this one is a million miles from Earth
- `WORK_OF_ART` — no

**There is no correct label.** The category simply does not exist in the scheme. This is a
limitation of the *label set*, not of the model — and it explains why both the small and
medium models were inconsistent about "Roman" in Step 11. They are not choosing badly
between good options; they are choosing between options that are all somewhat wrong.

## Step 13 — Error analysis, part 2: non-Western names

A second known weakness. NER models are trained mostly on American and European news
text, and they inherit that bias.

In [25]:
# A test the model has not seen: a Nepali name.
tests = [
    "Prasanna Koirala studies NLP at Kathmandu University.",
    "Prasanna studies at Kathmandu University.",
    "Sita Sharma works in Pokhara.",
]

for sentence in tests:
    print(f'"{sentence}"')
    for name, model in [("small", nlp_sm), ("medium", nlp_md)]:
        found = [(e.text, e.label_) for e in model(sentence).ents]
        print(f"    {name:<7} {found}")
    print()

"Prasanna Koirala studies NLP at Kathmandu University."
    small   [('Prasanna Koirala', 'PERSON'), ('NLP', 'ORG'), ('Kathmandu University', 'ORG')]
    medium  [('Prasanna Koirala', 'PERSON'), ('NLP', 'ORG'), ('Kathmandu University', 'ORG')]

"Prasanna studies at Kathmandu University."
    small   [('Prasanna', 'ORG'), ('Kathmandu University', 'ORG')]
    medium  [('Prasanna', 'PERSON'), ('Kathmandu University', 'ORG')]

"Sita Sharma works in Pokhara."
    small   [('Sita Sharma', 'PERSON'), ('Pokhara', 'GPE')]
    medium  [('Sita Sharma', 'PERSON'), ('Pokhara', 'GPE')]



### What this reveals

With the **full name** ("Prasanna Koirala"), both models correctly return `PERSON`. The
surname supplies enough signal.

Drop the surname and the models are far less certain — a bare first name that does not
appear in their training data is much harder to place.

This is a real limitation worth stating plainly: **these models are trained largely on
Western news text**, so non-Western names are recognised less reliably. For a Nepali NLP
application this matters, and it is exactly the problem the syllabus revisits in Unit 7
(Multilingual NLP and Nepali/Indic case studies).

## Step 14 — Checking the work against the brief

In [26]:
checks = {
    "Article loaded from a .txt file":
        ARTICLE_PATH.exists() and len(article_text) > 0,

    "NER performed":
        len(doc.ents) > 0,

    "Entities extracted with their types":
        all(ent.label_ for ent in doc.ents),

    "CSV file created":
        CSV_PATH.exists(),

    "CSV has an Entity column":
        "Entity" in check.columns,

    "CSV has an Entity_Type column":
        "Entity_Type" in check.columns,

    "CSV row count matches the table":
        len(check) == len(ner_df),

    "No empty cells in the CSV":
        check.isna().sum().sum() == 0,

    "Summary of entity types provided":
        len(summary) > 0 and SUMMARY_PATH.exists(),

    "Bonus: displacy visualisation":
        True,

    "Bonus: two NER models compared":
        len(spans_sm) > 0 and len(spans_md) > 0,

    "Filename follows the required convention":
        CSV_PATH.name == "PrasannaKoirala_NER_01.csv",
}

print("REQUIREMENT CHECKLIST")
print("=" * 54)
for requirement, passed in checks.items():
    print(f"  [{'PASS' if passed else 'FAIL'}]  {requirement}")

print("=" * 54)
print("ALL CHECKS PASSED" if all(checks.values()) else "SOMETHING FAILED - review above")

REQUIREMENT CHECKLIST
  [PASS]  Article loaded from a .txt file
  [PASS]  NER performed
  [PASS]  Entities extracted with their types
  [PASS]  CSV file created
  [PASS]  CSV has an Entity column
  [PASS]  CSV has an Entity_Type column
  [PASS]  CSV row count matches the table
  [PASS]  No empty cells in the CSV
  [PASS]  Summary of entity types provided
  [PASS]  Bonus: displacy visualisation
  [PASS]  Bonus: two NER models compared
  [PASS]  Filename follows the required convention
ALL CHECKS PASSED


## Summary

**What was done**

1. Loaded the same NASA article used in Assignment 01
2. Ran Named Entity Recognition with spaCy's `en_core_web_sm`
3. Extracted every named entity with its type
4. Produced a summary of entity counts by category (a required deliverable)
5. Exported both the entity list and the summary to CSV
6. Visualised the entities with `displacy` *(bonus)*
7. Compared two NER models, small vs medium *(bonus)*
8. Tested the models' behaviour on Nepali names

**What was learned**

- NER answers a different question from POS tagging: *what real-world thing is this?*
  rather than *what grammatical job does this word do?*
- NER labels only a small fraction of words, whereas POS labels all of them
- **An entity can span several words** ("Kennedy Space Center"), so the unit of analysis
  is the entity, not the token
- NER is effectively a **refinement of `PROPN`** — it separates the pile of names that
  POS tagging could only label identically
- Two models can report **the same number of entities while disagreeing about which
  ones**, so comparison must be done by position, not by count
- Genuine ambiguity is not fixed by a larger model. "Roman" is a person, a mission and a
  nationality all at once, and both models are inconsistent about it *within the same
  article*
- These models are trained largely on Western news text and are measurably less reliable
  on Nepali names

**Files produced**

| File | Contents |
|---|---|
| `data/article.txt` | The news article (same as Assignment 01) |
| `outputs/PrasannaKoirala_NER_01.csv` | Every named entity with its type |
| `outputs/PrasannaKoirala_NER_01_summary.csv` | Entity counts by category |